In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import cv2

seed =2019
np.random.seed(seed)
%matplotlib inline

Dosyanın bulundugu kisim

In [3]:
path =r"C:\Users\umuty\Desktop\Tipta_YZ_Gercek_Dunya_Projeleri\OCT_Goruntulerinden_Hastalik_Teshisi\OCT2017"
train_path = os.path.join(path, 'train') #egitim verisinin yolu
test_path = os.path.join(path, 'test') #test verisinin yolu
path,train_path,test_path


('C:\\Users\\umuty\\Desktop\\Tipta_YZ_Gercek_Dunya_Projeleri\\OCT_Goruntulerinden_Hastalik_Teshisi\\OCT2017',
 'C:\\Users\\umuty\\Desktop\\Tipta_YZ_Gercek_Dunya_Projeleri\\OCT_Goruntulerinden_Hastalik_Teshisi\\OCT2017\\train',
 'C:\\Users\\umuty\\Desktop\\Tipta_YZ_Gercek_Dunya_Projeleri\\OCT_Goruntulerinden_Hastalik_Teshisi\\OCT2017\\test')

ImageDataGenrator ile veri büyütme

Görüntüleri Aktarma (flow_from_directory)

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)  # %20 doğrulama için ayırıyoruz

train_generator = train_datagen.flow_from_directory(
    train_path, 
    target_size=(160, 160),  
    batch_size=16, 
    class_mode='categorical',  # 4 sınıf olduğu için categorical
    subset='training',  # %80 eğitim
    shuffle=True, # karıştırma işlemi
)

validation_generator = train_datagen.flow_from_directory(
    train_path, 
    target_size=(160, 160),  
    batch_size=16, 
    class_mode='categorical', 
    subset='validation',  # %20 doğrulama
    shuffle=True, # karıştırma işlemi
)


#shuffle: true verileri karıp overfitting olmasını engeller
#target_size : 150-200 arası hem bilgisayar açısından hem de model açısından yeterlidir

Found 66788 images belonging to 4 classes.
Found 16696 images belonging to 4 classes.


In [4]:
train_generator[0][0][0].shape

(160, 160, 3)

In [1]:
plt.imshow(train_generator[0][0][7])

NameError: name 'plt' is not defined

CallBacks

In [5]:
from keras.callbacks import ReduceLROnPlateau,ModelCheckpoint #öğrenme oranını azaltma ve belli aralıklarda ağırlık kaydetme

#val_loss'u izler, 5 epoch'ta bir öğrenme oranını %30 azaltır
lr = ReduceLROnPlateau(monitor = "val_loss", factor = 0.3, patience = 5, verbose =2, mode="max") 
#save_weights_only=True sadece ağırlığı kaydeder
os.makedirs("agirliklar", exist_ok=True)
cp = ModelCheckpoint(filepath = "agirliklar/eff_model.{epoch:02d}-{val_loss:.2f}.weights.h5",save_weights_only=True)

Transfer Learning ve Eğitim

In [6]:
from tensorflow.keras.applications import EfficientNetB5
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers

model = Sequential()
model.add(EfficientNetB5(weights = "imagenet", include_top = False, input_shape = (160,160,3)))
model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dropout(0.5))
model.add(layers.Dense(4, activation = "softmax"))

model.compile(optimizer = "adam", loss = "categorical_crossentropy", metrics = ["accuracy"])    

Eğitim

In [16]:
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,  # Artık test yerine doğrulama kullanılıyor
    callbacks=[lr, cp]
)
#####KASADA YAPILACAK

c:\Users\umuty\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


KeyboardInterrupt: 

Eğitimden Sonra Test Etme

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_path,  
    target_size=(150, 150),  
    batch_size=32,  
    class_mode='categorical',  # Çok sınıflı olduğu için categorical kullanıyoruz
    shuffle=False
)

# Modeli test veri kümesi üzerinde değerlendirme
test_loss, test_accuracy = model.evaluate(test_generator)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


In [7]:
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout,Input
from tensorflow.keras.layers import Conv2D,MaxPool2D,BatchNormalization

model_2 = Sequential([

    Conv2D(16,(3,3),activation='relu',input_shape=(160,160,3)),BatchNormalization(),MaxPool2D((2,2)),
    Conv2D(32,(3,3),activation='relu'),BatchNormalization(),MaxPool2D((2,2)),
    Conv2D(64,(3,3),activation='relu'),BatchNormalization(),MaxPool2D((2,2)),
    Conv2D(128,(3,3),activation='relu'),BatchNormalization(),MaxPool2D((2,2)),Dropout(0.2),
    Conv2D(256,(3,3),activation='relu'),BatchNormalization(),MaxPool2D((2,2)),Dropout(0.2),
    
    Flatten(),
    Dense(1024,activation='relu'),
    Dropout(0.3),
    Dense(4,activation='softmax')
    ])

c:\Users\umuty\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
model_2.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [10]:
model_2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 158, 158, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 158, 158, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 79, 79, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 77, 77, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 77, 77, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 38, 38, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 36, 36, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 36, 36, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 18, 18, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 6, 6, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 6, 6, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1024)           │     2,360,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │         4,100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,759,012 (10.52 MB)

 Trainable params: 2,758,020 (10.52 MB)

 Non-trainable params: 992 (3.88 KB)

EĞİTİM

In [20]:
history_cnn = model_2.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,  # Artık test yerine doğrulama kullanılıyor
    batch_size=16
)
#####KASADA YAPILACAK

Epoch 1/10
 121/4175 ━━━━━━━━━━━━━━━━━━━━ 12:14 181ms/step - accuracy: 0.4676 - loss: 2.8673

KeyboardInterrupt: 

Accuracy-Loss Grafiği

In [ ]:
plt.plot(history_cnn.history['accuracy'], label='train accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='validation accuracy')
plt.legend("Eğitim","Doğrulama") #gösterge
plt.title("Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")

NameError: name 'history_cnn' is not defined

In [ ]:
plt.plot(history_cnn.history['loss'], label='train loss')
plt.plot(history_cnn.history['val_loss'], label='validation loss')
plt.legend("Eğitim","Doğrulama") #gösterge
plt.title("Loss")
plt.xlabel("Epochs")

Hata Matrisi (Confusion Matrix)

In [11]:
test_path

'C:\\Users\\umuty\\Desktop\\Tipta_YZ_Gercek_Dunya_Projeleri\\OCT_Goruntulerinden_Hastalik_Teshisi\\OCT2017\\test'

In [12]:
dosya = os.listdir(test_path)
dosya

['CNV', 'DME', 'DRUSEN', 'NORMAL']

In [13]:
import glob
test_data = []
test_label = []

for sinif in dosya:
    print(sinif)
    dosya_yolu = os.path.join(test_path,sinif)
    print(dosya_yolu)
    for img in glob.glob(dosya_yolu + "/*.jpeg"):
        img = cv2.imread(img)
        img = cv2.resize(img,(160,160))
        img = img.astype('float32') #tensorflow bunu kullandığı için burda da float32 yapıyoruz

        if sinif == "CNV":
            label = 0
        elif sinif == "DME":
            label = 1
        elif sinif == "DRUSEN":
            label = 2
        elif sinif == "NORMAL":
            label = 3
        test_data.append(img)
        test_label.append(label)
        print(len(test_data))

CNV
C:\Users\umuty\Desktop\Tipta_YZ_Gercek_Dunya_Projeleri\OCT_Goruntulerinden_Hastalik_Teshisi\OCT2017\test\CNV
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249

In [14]:
test_data[0].shape

(160, 160, 3)

In [16]:
x_test = np.asarray(test_data) #içine yazdığımız verileri numpy dizisine çeviriyoruz
x_test.shape

(1000, 160, 160, 3)

In [20]:
y_test = np.asarray(test_label) 
y_test.shape


(1000,)

HATA MATRİSİ (CONFUSİON MATRİX)

y_true : Doktor tarafından belirlenmiş sonuçlar
y_pred : modelin tahmin ettiği sonuçlar (veriler sınıf şekilindedir)  model.predict_classes() ile elde edilebilir
y_probas : modelin tahmin ettiği sonuçlar (veriler olasılık şeklindedir) model.predict ile elde edilebilir

In [21]:
y_true = y_test
y_pred = model.predict_classes(x_test) #modelin tahmin ettiği sınıflar
y_probas = model.predict(x_test) #modelin tahmin ettiği sınıfların olasılıkları

AttributeError: 'Sequential' object has no attribute 'predict_classes'

In [ ]:
y_true[0], y_pred[0], y_probas[0] #ilk elemanları kontrol ediyoruz

In [22]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

In [ ]:
cm = confusion_matrix(y_true, y_pred)

In [ ]:
cm

In [ ]:
sns.heatmap(cm)

In [ ]:
plt.figure(figsize=(6, 5)) #tamn bir kare olarak gözükmesini sağlar
sns.heatmap(cm)

In [ ]:
plt.figure(figsize=(6, 5)) #tamn bir kare olarak gözükmesini sağlar
sns.heatmap(cm,annot=True) #annot=True (sayıları gösterir-gösterge)

In [ ]:
plt.figure(figsize=(6, 5)) #tamn bir kare olarak gözükmesini sağlar
sns.heatmap(cm,annot=True,fmt="n") #annot=True (sayıları gösterir-gösterge)  fmt="n" (format :tam sayı gösterir)

In [ ]:
cm2 = pd.DataFrame(cm, columns = ["CNV","DME","DRUSEN","NORMAL"], index = ["CNV","DME","DRUSEN","NORMAL"])
cm2

In [ ]:
plt.figure(figsize=(6, 5)) #tamn bir kare olarak gözükmesini sağlar
sns.heatmap(cm2,annot=True,fmt="n") #annot=True (sayıları gösterir-gösterge)  fmt="n" (format :tam sayı gösterir)

Aşağıdaki kodla altta yazan sınıfları üst tarafa alıyoruz

In [ ]:
plt.figure(figsize=(6, 5)) #tamn bir kare olarak gözükmesini sağlar
hm = sns.heatmap(cm2,annot=True,fmt="n") #annot=True (sayıları gösterir-gösterge)  fmt="n" (format :tam sayı gösterir)
#üstteki etiketleri gösterir alttakileri gizler ve yazıların altındaki tickleri (siyah çizgileri) üste alır alttan kaldırır
hm.tick_params(labeltop=True, labelbottom=False,top=True, bottom=False)

hm.set_xlabel("Tahmin Edilen Sınıflar")
hm.xaxis_set_label_position("top") #basligin ust tarafında gözükmesini sağlar
hm.set_ylabel("Gerçek Sınıflar")

Renklerle falan oynuyoruz

In [ ]:
plt.figure(figsize=(6, 5)) #tamn bir kare olarak gözükmesini sağlar
hm = sns.heatmap(cm2,annot=True,fmt="n",cmap="Blues") #annot=True (sayıları gösterir-gösterge)  fmt="n" (format :tam sayı gösterir)
#üstteki etiketleri gösterir alttakileri gizler ve yazıların altındaki tickleri (siyah çizgileri) üste alır alttan kaldırır
hm.tick_params(labeltop=True, labelbottom=False,top=True, bottom=False)

hm.set_xlabel("Tahmin Edilen Sınıflar")
hm.xaxis_set_label_position("top") #basligin ust tarafında gözükmesini sağlar
hm.set_ylabel("Gerçek Sınıflar")

ROC EĞRİSİ

Gerçek Pozitif (True Positive -TP): Kişi hasta ve hasta olarak doğru teşhis edilmiş
Yanlış Pozitif (False Positive -FP): Kişi sağlıklı ama hasta olarak yanlış teşhis edilmiş
Gerçek Negatif (True Negative -TN): Kişi sağlıklı ve sağlıklı olarak doğru teşhis edilmiş
Yanlış Negatif (False Negative -FN): Kişi hasta ama sağlıklı olarak yanlış teşhis edilmiş

Y ekseninde TP oranı (Sensivite Hassasiyet)
X ekseninde FP oranı (1-Spesifite)
Bu eğrinin altında kalan alan AUC olarak adlandırlır
AUC ve Accuracy aynı şey değil !!!!!
Genelde ikili sınıflandırmada kullanılır

In [ ]:
#scikit-learn kutuphanesi sadece binasry oldugu icin bu kutuphaneyi kullanıyoruz
import scikitplot as skplt   
skplt.metrics.plot_roc(y_true, y_probas, plot_micro=False, plot_macro=False)

ingilizce başlıkları türkçeye çevirelim

In [ ]:
skplt.metrics.plot_roc(y_true, y_probas, plot_micro=False, plot_macro=False)
plt.title("ROC Eğrisi")
plt.xlabel("False Positive(FP) Oranı")
plt.ylabel("True Positive(TP) Oranı")
plt.legend(["CNV için ROC Eğrisi (area=1.00)",
            "DME için ROC Eğrisi (area=1.00)" ,
            "DRUSEN için ROC Eğrisi (area=1.00)" ,
            "NORMAL için ROC Eğrisi (area=1.00)"],loc= "lower right") #içerdeki gösterge isimlerini değiştiriyoruz ve konumu sağ alt yapıyoruz

plt.show()
